# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the [FAIR² Croissant dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) using the `mlcroissant` library.

### Dataset Source
This dataset details high-quality protein abundance, modification, and sequence data for human samples—collected via mass spectrometry after stimulating human mast cells and isolating extracellular vesicles. The dataset follows the Croissant standard and is enriched with comprehensive structural metadata.

In [ ]:
# Ensure `mlcroissant` library is installed in this environment
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)

# Print basic metadata
metadata_dict = json.loads(dataset.metadata.json())  # serialize metadata for display
title = metadata_dict.get('name')
description = metadata_dict.get('description')
print(f"{title}: {description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Below, we identify the record sets in this dataset and output their metadata. We then inspect the fields (columns) in each record set, referencing all entities by their unique `@id`.

In [ ]:
# Explore the dataset structure

# Get all record set objects
record_sets = list(dataset.metadata.record_sets)

print("Available record sets and their identifiers:\n")
for rs in record_sets:
    print(f"@id: {rs.id}\n  name: {rs.name}\n  description: {rs.description}\n")

print("---\nFields in each Record Set:\n")
for rs in record_sets:
    print(f"Record Set '@id': {rs.id}  | name: {rs.name}")
    for field in rs.fields:
        field_id = field.id
        dtype = field.data_type
        print(f"  Field @id: {field_id}  | name: {field.name}  | data_type: {dtype}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

Choose a relevant record set and use its `@id` for extraction. Optionally, extract all record sets and preview their content.

In [ ]:
# Build a list of record set @ids
record_set_ids = [rs.id for rs in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:  # Only store if there are rows
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded {len(records)} rows for record set {record_set_id}")
    else:
        print(f"No records found for record set {record_set_id}")

# For demonstration, pick the main protein record set (identified by '@id')
# If only one record set is present (common for flat tables), use it. 
if len(dataframes) == 0:
    raise ValueError("No data could be loaded from any record set. Check the dataset or schema.")
main_record_set_id = list(dataframes.keys())[0]
print(f"\nColumn preview for record set: {main_record_set_id}\n")
print(dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply basic analysis: filtering records by a numeric field, creating a normalized version, and grouping.

> ⚠️ **Note:** Use field `@id`s for column names below. If uncertain, inspect columns above and adjust as needed.

In [ ]:
# --- Identify a numeric field by '@id' ---
# Here we try 'cr:peptide_count' or 'cr:abundance' as typical mass spec output fields.
# Inspect exact @id and replace in variable below as needed.

df = dataframes[main_record_set_id]
numeric_field_id = None
group_field_id = None

# Try to auto-select a numeric field and a categorical field
# Use available field IDs and pick typical quantitative fields
for col in df.columns:
    lowercol = col.lower()
    if (('abundance' in lowercol or 'count' in lowercol or 'coverage' in lowercol or 'mw' in lowercol or 'pi' in lowercol) and
        pd.api.types.is_numeric_dtype(df[col])):
        numeric_field_id = col
        break
        
# Try to pick a categorical field, often 'cr:sample' or 'cr:description' or 'cr:protein'
for col in df.columns:
    lowercol = col.lower()
    if (('sample' in lowercol or 'group' in lowercol or 'type' in lowercol or 'description' in lowercol) and
        pd.api.types.is_string_dtype(df[col])):
        group_field_id = col
        break

if not numeric_field_id:
    raise ValueError("Could not determine a numeric field in the selected record set.")

print(f"Using numeric field '@id': {numeric_field_id}")
if group_field_id:
    print(f"Using group field '@id': {group_field_id}\n")
else:
    print("No group field could be determined. Group analysis will be skipped.\n")

# Example: filter numeric > threshold
threshold = df[numeric_field_id].quantile(0.75)  # use top quartile as threshold for illustration
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold}")
display(filtered_df[[numeric_field_id]].head())

# Normalize the numeric field
filtered_df[numeric_field_id + '_normalized'] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
display(filtered_df[[numeric_field_id, numeric_field_id + '_normalized']].head())

# Optional: group by category and take mean
if group_field_id:
    grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
    print(f"Grouped mean values by {group_field_id}:")
    display(grouped_df.head())

## 5. Visualization
Visualize the numeric field distribution and the result of grouping (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot the distribution of the numeric field
plt.figure(figsize=(8,4))
sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=30)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Frequency")
plt.show()

# If grouping was done, plot group means
if group_field_id:
    plt.figure(figsize=(12,5))
    grouped_df[numeric_field_id].plot(kind='bar')
    plt.title(f"Mean {numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(f"Mean {numeric_field_id}")
    plt.tight_layout()
    plt.show()

## 6. Conclusion

In this notebook, we've demonstrated how to:
- Load a Croissant-compliant biomedical dataset using `mlcroissant`
- Inspect available record sets and their field `@id`s
- Extract and process tabular data for data science workflows
- Perform simple analyses: filtering, normalization, grouping
- Visualize quantitative and grouped results

You can further expand the analysis by exploring other record sets, joining data where appropriate, and building advanced visualizations or machine learning pipelines tailored to your bioinformatics questions.